# nano Beatrix v1 — the output layer on AdamW: the lead body and Castor's second seed

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_head_decay_lead_s4.ipynb)

**Why this runs.** `nano_v1_head_decay.ipynb` moves the output layer's two readout matrices (`head.w_h`, `head.w_s`) from Muon to AdamW on Castor's seed-3 start, at decay 0, 0.1 and 1.2, to stop the readout's growth that the last block's big direction answers (the reasons and the reads are in that notebook). Three of the four v1 crafts are Pollux-layout bodies, and a claim about a fix needs a second seed, so the same change runs here on:

| arm | body | start | the output layer's readout | its plain run (the reference) |
|---|---|---|---|---|
| `pdc-head-adamw0.1` | Pollux's layout, dual hubs + Canon (the lead body) | the v1 start (seed 1) | AdamW, decay 0.1 | `pdc-bare` |
| `c-head-adamw0.1-s4` | Castor | seed 4 | AdamW, decay 0.1 | `c-bare-s4` |
| `pdc-head-adamw1.2` | the lead body | seed 1 | AdamW, decay 1.2 | `pdc-bare` |
| `c-head-adamw1.2-s4` | Castor | seed 4 | AdamW, decay 1.2 | `c-bare-s4` |

Everything else is the 2s recipe (Muon 0.02 on the other matrices, Adam 0.0003 without decay on the rest, 200 warmup steps then flat, clip 1.0, 262,144 bytes a step), the same data in the same order as the first stage, 4,960 steps. Decay 0.1 is the common setting; 1.2 is 0.1's total pull over the full 61,422-step diet, within these 4,960 steps. Each arm starts from its plain run's start, bit for bit; the plain runs are read from the repo (`pdc-bare` comes from the first stage's notebook).

**How an arm passes** (written before any arm runs; the second stage's rule, each arm against its own plain run): (i) the share on the last block's direction stops climbing after the warmup (at 4,960 no more than 0.01 above its own value at 1,500) and sits under its plain run's at every read from 1,500 on; (ii) bits a byte at 4,960 within 0.0029 of its plain run (the spread of Castor's two plain seeds, the only measured seed spread; at least 0.001); (iii) the exit's H on random bytes inside the plain runs' spread. The output layer's size and loudness (`lastwrite/head_*`) show whether an arm acts by stopping the readout's growth.

**Length:** about 46 minutes a lead-body arm and 30 a Castor arm on an RTX PRO 6000: about 2.5 hours for the four; the cell can be stopped and rerun (each arm resumes from the repo, finished arms are read back).

**Token:** `HF_TOKEN` with write access to AbstractPhil/alephllm-nano-beatrix-training (the records land under `v1_lastfix/<arm>/`, beside the first two notebooks').

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.30"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import lastfix_screen as LS
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 30), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK ============
OUT = '/content/nano_v1_head_decay_lead_s4'           # the session folder: the starts, the runs, the records
REPO = LS.REPO                                        # AbstractPhil/alephllm-nano-beatrix-training, under v1_lastfix/<arm>/
NAMES = ['pdc-head-adamw0.1', 'c-head-adamw0.1-s4', 'pdc-head-adamw1.2', 'c-head-adamw1.2-s4']
REFS = ['pdc-bare', 'c-bare-s4', 'c-bare-s3']         # the plain runs (read from the repo, never retrained here)
MAX_HOURS = None                                      # this session's budget in hours (None: until every arm is done)
print('arms:', ', '.join(NAMES), '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: every arm's start (checked against a fresh build), its optimizer split, the length, the push probe. Trains nothing.
import json, time
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.train.optim import build_optimizers
births = {n: LS.write_arm_birth(LS.BY_NAME[n], f'{OUT}/births') for n in NAMES}
step_s = {'nano-castor-v1': 0.299, 'nano-pollux-dual-canon-v1': 0.493}      # the v1 bench on an RTX PRO 6000 at 16 x 4 (2026-10-14)
for n in NAMES:
    a = LS.BY_NAME[n]
    t = LS.arm_preset(a, births[n]).train
    m = AlephLM(LS.arm_cfg(a))
    opts = build_optimizers(m, t.muon_lr, t.muon_momentum, t.adam_lr, decay_prefixes=t.decay_params, decay_wd=t.decay_wd)
    g = opts[1].param_groups[1]
    print(f"{n}: {a['craft']} seed {a['seed']} (plain run {LS.PLAIN[(a['craft'], a['seed'])]}); {sum(p.numel() for p in g['params']):,} "
          f"readout weights on {type(opts[1]).__name__} at decay {g['weight_decay']}; {sum(p.numel() for p in opts[0].param_groups[0]['params']):,} on Muon")
mins = sum((4960 * step_s[LS.BY_NAME[n]['craft']] + 12 * 25) / 60 for n in NAMES)
print(f'-> about {mins / 60:.1f} h on an RTX PRO 6000 (training plus a full eval and the gauge every 500 steps)')
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = f'{LS.PREFIX}/preflight/push_probe_head_decay_lead_s4.json'
    api.upload_file(path_or_fileobj=json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                                                'card': torch.cuda.get_device_name(0), 'arms': NAMES}).encode(),
                    path_in_repo=path, repo_id=REPO, commit_message='v1 head-decay carry-over: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: C3 runs locally and ships nothing')

In [ ]:
# C3 (T) — THE SCREEN: arm by arm; the trainer's own lines and the screen's gauge lines ([lastfix]) print as they come. Rerun the cell to
# continue (a new session too): each arm resumes from the repo, finished arms are read back. The stop button saves the running arm first.
RECS = LS.run(OUT, HF_TOKEN if PUSH_OK else None, NAMES, max_hours=MAX_HOURS)
from huggingface_hub import hf_hub_download
from IPython.display import Markdown, display

def with_refs(recs):
    out = {}
    for n in REFS:
        try:
            out[n] = json.load(open(hf_hub_download(REPO, f'{LS.PREFIX}/{n}/screen_record.json', token=HF_TOKEN or False), encoding='utf-8'))
        except Exception as e:  # noqa: BLE001
            print('no record yet for', n, type(e).__name__)
    out.update(recs)
    return out
display(Markdown('**At 1,500 (after the warmup)**\n\n' + LS.table(with_refs(RECS), 1500)))
display(Markdown('**At the end (each arm\'s last read)**\n\n' + LS.table(with_refs(RECS))))

In [ ]:
# C4 — THE CURVES AND THE TABLES from the repo (every session; tokenless: this session's files). Trains nothing.
import json
from IPython.display import Markdown, display
recs, logdir = {}, f'{OUT}/runs'
if HF_TOKEN:
    from huggingface_hub import snapshot_download
    hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub',
                            allow_patterns=[f'{LS.PREFIX}/{n}/runs/*' for n in REFS + NAMES] + [f'{LS.PREFIX}/{n}/screen_record.json' for n in REFS + NAMES])
    logdir = f'{hub}/{LS.PREFIX}'
    for n in REFS + NAMES:
        p = f'{hub}/{LS.PREFIX}/{n}/screen_record.json'
        if os.path.exists(p):
            recs[n] = json.load(open(p, encoding='utf-8'))
else:
    for n in NAMES:
        p = f'{OUT}/lastfix/{n}.json'
        if os.path.exists(p):
            recs[n] = json.load(open(p, encoding='utf-8'))
display(Markdown('**At 1,500 (after the warmup)**\n\n' + LS.table(recs, 1500)))
display(Markdown('**Each arm\'s last read**\n\n' + LS.table(recs)))
%load_ext tensorboard
%tensorboard --logdir {logdir}

### Reading the result

Each arm is read against its own plain run (the table's "vs its plain run" column): the lead-body arms against `pdc-bare`, the
seed-4 arms against `c-bare-s4`.

| read | the plain run | an arm that works |
|---|---|---|
| the readout's size (`lastwrite/head_w_h_fro`) | grows through the run | grows less, or settles |
| share on the last block's direction | climbs after the warmup | flat or falling after the warmup, under its plain run at every read |
| bits a byte at 4,960 | the reference | within 0.0029 |
| the exit's H on random bytes | the reference | inside the plain runs' spread |

### Changing the screen
| to | the edit in C1 |
|---|---|
| the lead body only | `NAMES = [n for n in NAMES if n.startswith('pdc-')]` |
| a session budget | `MAX_HOURS = 1` |